# 選挙マニフェスト超党派政策NLP解析

衆議院・参議院のマニフェストを分析して、
1. 超党派で重視されている政策テーマを抽出
2. 政策ベクトルをクラスタリング
3. 2Dマップへマッピング
を実施します。

## Section 0: セットアップとインポート

In [ ]:
import sys
import os

# パス設定
base_path = '/c/GitHub/Data-Analysis_competition/SSDSE'
election_path = os.path.join(base_path, 'election')
data_path = os.path.join(base_path, 'data')
src_path = os.path.join(election_path, 'src')
results_path = os.path.join(src_path, 'results')  # src/results に変更

sys.path.insert(0, src_path)

# ディレクトリ確認
print(f"Base path: {base_path}")
print(f"Data path exists: {os.path.exists(data_path)}")
print(f"Results path exists: {os.path.exists(results_path)}")

In [ ]:
# 必須ライブラリのインポート
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import re
from tqdm import tqdm
from collections import Counter

# 自作モジュール
from text_cleaner import clean_manifesto_text, split_into_paragraphs, normalize_party_name, quality_report
from tokenizer import JapaneseTokenizer, token_frequency

# NLPライブラリ
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import umap

# 可視化
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 警告抑制
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定
matplotlib.rcParams['font.family'] = 'Meiryo'
matplotlib.rcParams['axes.unicode_minus'] = False

print("✓ 全ライブラリインポート完了")

In [ ]:
# 選挙結果データ（議席数）
# 衆議院
seats_shu = {
    '自民党': 316,
    '中道': 49,
    '日本維新の党': 36,
    '国民民主党': 28,
    '参政党': 15,
    'チームみらい': 11,
    '共産党': 4,
    '令和新選組': 1,
    '日本保守党': 0,
    '社民党': 0,
}

# 参議院
seats_san = {
    '自民党': 39,
    '立憲民主党': 22,
    '国民民主党': 17,
    '参政党': 14,
    '公明党': 8,
    '日本維新の党': 7,
    '令和新選組': 3,
    '共産党': 3,
    '日本保守党': 2,
    '社民党': 1,
    'チームみらい': 1,
}

print("✓ 選挙結果データ設定完了")

## Section 1: データ読み込みと前処理

In [ ]:
def load_and_clean_data():
    """
    衆議院・参議院のマニフェストテキストを読み込んで前処理
    """
    data_records = []

    # 衆議院
    shu_files = glob.glob(os.path.join(data_path, '衆議院', '*.txt'))
    for file_path in shu_files:
        party_name = Path(file_path).stem
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_text = f.read()
        except:
            with open(file_path, 'r', encoding='utf-8-sig') as f:
                raw_text = f.read()

        # 正規化
        party_name = normalize_party_name(party_name)

        # クリーニング
        cleaned_text = clean_manifesto_text(raw_text, party_name, '衆議院')

        data_records.append({
            'party': party_name,
            'chamber': '衆議院',
            'raw_text': raw_text,
            'cleaned_text': cleaned_text,
            'file_path': file_path
        })

    # 参議院
    san_files = glob.glob(os.path.join(data_path, '参議院', '*.txt'))
    for file_path in san_files:
        party_name = Path(file_path).stem
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_text = f.read()
        except:
            with open(file_path, 'r', encoding='utf-8-sig') as f:
                raw_text = f.read()

        party_name = normalize_party_name(party_name)
        cleaned_text = clean_manifesto_text(raw_text, party_name, '参議院')

        data_records.append({
            'party': party_name,
            'chamber': '参議院',
            'raw_text': raw_text,
            'cleaned_text': cleaned_text,
            'file_path': file_path
        })

    df_documents = pd.DataFrame(data_records)
    return df_documents

print("Loading manifesto data...")
df_documents = load_and_clean_data()
print(f"✓ {len(df_documents)} 文書を読み込み")
print(f"\n政党一覧:")
print(df_documents[['party', 'chamber']].drop_duplicates().sort_values('party'))

In [ ]:
# パラグラフへの分割
paragraph_records = []

for idx, row in df_documents.iterrows():
    paragraphs = split_into_paragraphs(row['cleaned_text'], min_length=30)
    for p_idx, para in enumerate(paragraphs):
        paragraph_records.append({
            'party': row['party'],
            'chamber': row['chamber'],
            'para_idx': p_idx,
            'text': para
        })

df_paragraphs = pd.DataFrame(paragraph_records)
print(f"✓ {len(df_paragraphs)} パラグラフに分割")
print(f"\nパラグラフ統計:")
print(f"政党数: {df_paragraphs['party'].nunique()}")
print(f"議院別: 衆議院 {(df_paragraphs['chamber']=='衆議院').sum()}, 参議院 {(df_paragraphs['chamber']=='参議院').sum()}")

In [ ]:
# データ品質レポート
quality_report(df_paragraphs)

## Section 2: 日本語トークナイゼーション

In [ ]:
# トークナイザーの初期化
tokenizer = JapaneseTokenizer()
print("✓ JapaneseTokenizer 初期化完了")

# テスト: サンプルテキストをトークナイズ
sample_text = df_paragraphs.iloc[0]['text']
print(f"\nサンプルテキスト:")
print(f"{sample_text[:100]}...")

sample_tokens = tokenizer.tokenize(sample_text)
print(f"\nトークン数: {len(sample_tokens)}")
print(f"トークン: {sample_tokens[:20]}")

In [ ]:
# 全パラグラフのトークナイゼーション
print("Tokenizing all paragraphs...")
df_paragraphs['tokens'] = df_paragraphs['text'].apply(lambda x: tokenizer.tokenize(x))
df_paragraphs['token_count'] = df_paragraphs['tokens'].apply(len)

# トークン化されたテキスト（スペース区切り）も作成
df_paragraphs['tokenized_text'] = df_paragraphs['tokens'].apply(lambda x: ' '.join(x))

print(f"✓ 全パラグラフのトークナイゼーション完了")
print(f"\n平均トークン数: {df_paragraphs['token_count'].mean():.1f}")
print(f"トークン数の分布: min={df_paragraphs['token_count'].min()}, max={df_paragraphs['token_count'].max()}")

In [ ]:
# 政党別トークン統計
token_stats = df_paragraphs.groupby('party').agg({
    'token_count': ['sum', 'mean', 'count']
}).round(1)
token_stats.columns = ['total_tokens', 'avg_tokens_per_para', 'para_count']
token_stats = token_stats.sort_values('total_tokens', ascending=False)

print("\n政党別トークン統計:")
print(token_stats)

In [ ]:
# 全体の頻出トークン（ストップワード除去後）
all_tokens = []
for tokens_list in df_paragraphs['tokens']:
    all_tokens.extend(tokens_list)

top_tokens = Counter(all_tokens).most_common(30)
print("\n全体の頻出トークン (top 30):")
for token, count in top_tokens:
    print(f"  {token}: {count}")

## Section 3: TF-IDF + NMF トピックモデリング

In [ ]:
# 政党レベルの統合テキスト（TF-IDF用）
party_documents = df_documents.copy()
party_documents['tokenized_text'] = party_documents['cleaned_text'].apply(
    lambda x: ' '.join(tokenizer.tokenize(x))
)

print(f"✓ {len(party_documents)} 党のドキュメントを準備")
print(f"\n政党別トークン数（党レベル）:")
party_documents['token_count'] = party_documents['tokenized_text'].apply(lambda x: len(x.split()))
print(party_documents[['party', 'chamber', 'token_count']].to_string())

In [ ]:
# TF-IDF ベクトライザ
print("\nFitting TfidfVectorizer...")
tfidf = TfidfVectorizer(
    min_df=2,           # 最低2つの文書で出現
    max_df=0.9,         # 最大90%の文書で出現
    ngram_range=(1, 2), # 1-gramと2-gram
    sublinear_tf=True   # 文書長の不均一を補正
)

# 政党レベルのTF-IDF行列
X_tfidf_party = tfidf.fit_transform(party_documents['tokenized_text'])

print(f"✓ TF-IDF 行列作成: {X_tfidf_party.shape}")
print(f"  (政党数: {X_tfidf_party.shape[0]}, 語彙数: {X_tfidf_party.shape[1]})")

In [ ]:
# NMF でトピック抽出
n_topics = 12
print(f"\nFitting NMF with {n_topics} topics...")

nmf = NMF(
    n_components=n_topics,
    init='nndsvda',
    random_state=42,
    max_iter=500
)

W = nmf.fit_transform(X_tfidf_party)  # 政党-トピック行列 (19 x 12)
H = nmf.components_                    # トピック-単語行列 (12 x vocab)

print(f"✓ NMF フィッティング完了")
print(f"  W (政党-トピック): {W.shape}")
print(f"  H (トピック-単語): {H.shape}")

In [ ]:
# トピックの上位単語を表示
vocab = tfidf.get_feature_names_out()

print("\n" + "="*80)
print("トピック-単語行列 (各トピックの上位15単語)")
print("="*80)

for topic_id in range(n_topics):
    top_indices = H[topic_id].argsort()[-15:][::-1]
    top_words = vocab[top_indices]
    top_weights = H[topic_id][top_indices]

    print(f"\nTopic {topic_id}:")
    for word, weight in zip(top_words, top_weights):
        print(f"  {word:15} {weight:.3f}")

In [ ]:
# トピックラベルを手動で定義（上記の単語分析に基づいて）
# NOTE: これは目視で各トピックを確認した後に設定してください
topic_labels = {
    0: '経済・成長戦略',
    1: '社会保障・年金',
    2: '教育・子育て支援',
    3: 'エネルギー・環境',
    4: '外交・安全保障',
    5: '医療・介護',
    6: '税制改革',
    7: 'デジタル・行政改革',
    8: '地方創生',
    9: '労働・雇用',
    10: '農業・食糧安保',
    11: '憲法・政治改革',
}

print("✓ トピックラベルを設定")
print(f"\nトピック一覧:")
for topic_id, label in topic_labels.items():
    print(f"  Topic {topic_id}: {label}")

In [ ]:
# 超党派重要度スコアの計算

# 各議院の総議席数
total_seats_shu = sum(seats_shu.values())
total_seats_san = sum(seats_san.values())

# 政党ごとの総議席数（衆参合計の重み）
party_seats = {}
for party in party_documents['party'].unique():
    seats = seats_shu.get(party, 0) + seats_san.get(party, 0)
    party_seats[party] = seats

# 議席数ベクトル（NMFの政党順）
seats_vector = np.array([party_seats.get(party, 0) for party in party_documents['party']])
total_seats = seats_vector.sum()

# 議席数加重スコア
salience_weighted = (W * seats_vector.reshape(-1, 1)).sum(axis=0) / total_seats

# 出現政党数スコア（重みが0.1以上の政党数）
salience_count = (W >= 0.1).sum(axis=0)

# トピックスコア DataFrame
topic_scores = pd.DataFrame({
    'topic_id': range(n_topics),
    'topic_label': [topic_labels[i] for i in range(n_topics)],
    'salience_weighted': salience_weighted,
    'party_count': salience_count,
})
topic_scores = topic_scores.sort_values('salience_weighted', ascending=False)

print("\n超党派重要度スコア:")
print(topic_scores.to_string(index=False))

In [ ]:
# 政党-トピック行列を DataFrame に
party_topic_matrix = pd.DataFrame(
    W,
    index=party_documents['party'].values,
    columns=[topic_labels[i] for i in range(n_topics)]
)

# 結果をCSVに保存
topic_scores.to_csv(os.path.join(results_path, 'ranked_topics.csv'), index=False, encoding='utf-8')
party_topic_matrix.to_csv(os.path.join(results_path, 'party_topic_matrix.csv'), encoding='utf-8')

print(f"\n✓ 結果を {results_path} に保存")
print("  - ranked_topics.csv")
print("  - party_topic_matrix.csv")

## Section 4: センテンス埋め込みとクラスタリング

In [ ]:
# センテンス埋め込みモデルのロード
print("Loading sentence-transformer model...")
embedding_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
print("✓ モデルロード完了")

In [ ]:
# パラグラフの埋め込み
print(f"\nEmbedding {len(df_paragraphs)} paragraphs...")
embeddings = embedding_model.encode(
    df_paragraphs['text'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"✓ 埋め込み完了: {embeddings.shape}")

# キャッシュに保存
np.save(os.path.join(results_path, 'paragraph_embeddings.npy'), embeddings)
print(f"✓ 埋め込みをキャッシュに保存")

In [ ]:
# PCA で次元削減（ノイズ低減 & 速度向上）
print("\nApplying PCA...")
pca = PCA(n_components=50, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

print(f"✓ PCA 完了: {embeddings_pca.shape}")
print(f"  説明分散: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# K-Means クラスタリング
print(f"\nApplying K-Means (k={n_topics})...")
kmeans = KMeans(
    n_clusters=n_topics,
    random_state=42,
    n_init=10
)
cluster_labels = kmeans.fit_predict(embeddings_pca)

df_paragraphs['cluster'] = cluster_labels

# シルエットスコア（品質指標）
from sklearn.metrics import silhouette_score
silhouette = silhouette_score(embeddings_pca, cluster_labels)

print(f"✓ クラスタリング完了")
print(f"  Silhouette Score: {silhouette:.3f}")
print(f"\nクラスタサイズ:")
print(df_paragraphs['cluster'].value_counts().sort_index())

In [ ]:
# 政党別の平均埋め込み
print("\nCalculating party-level embeddings...")
party_embeddings_dict = {}

for party in df_paragraphs['party'].unique():
    party_mask = df_paragraphs['party'] == party
    party_emb = embeddings[party_mask].mean(axis=0)
    party_embeddings_dict[party] = party_emb

# 政党リストと対応する埋め込み（保存用）
party_list = sorted(party_embeddings_dict.keys())
party_embeddings_array = np.array([party_embeddings_dict[p] for p in party_list])

print(f"✓ {len(party_list)} 政党の平均埋め込みを計算")

In [ ]:
# パラグラフクラスタ結果の保存
df_paragraphs_export = df_paragraphs[['party', 'chamber', 'text', 'token_count', 'cluster']].copy()
df_paragraphs_export['cluster_label'] = df_paragraphs_export['cluster'].map(
    {i: topic_labels[i] for i in range(n_topics)}
)
df_paragraphs_export.to_csv(os.path.join(results_path, 'paragraph_clusters.csv'), index=False, encoding='utf-8')

print(f"✓ パラグラフクラスタ結果を保存")

## Section 5: UMAP + 可視化

In [ ]:
# UMAP で2D削減（パラグラフ）
print("Applying UMAP (paragraphs)...")
reducer = umap.UMAP(
    n_neighbors=10,
    min_dist=0.15,
    metric='cosine',
    random_state=42
)

umap_para = reducer.fit_transform(embeddings)
print(f"✓ UMAP 完了: {umap_para.shape}")

# UMAP で2D削減（政党）
umap_parties = reducer.transform(party_embeddings_array)
print(f"✓ 政党のUMAP変換完了: {umap_parties.shape}")

In [ ]:
# UMAP座標をパラグラフDFに追加
df_paragraphs['umap_x'] = umap_para[:, 0]
df_paragraphs['umap_y'] = umap_para[:, 1]

# 政党のイデオロギー位置を定義（簡易版）
# 左: 共産党, 立憲民主党, 社民党, 令和新選組
# 中: 公明党, 国民民主党, チームみらい
# 右: 自民党, 維新, 参政党, 保守党, 中道

ideology_map = {
    '共産党': 'left',
    '立憲民主党': 'left',
    '社民党': 'left',
    '令和新選組': 'left',
    '公明党': 'center',
    '国民民主党': 'center',
    'チームみらい': 'center',
    '自民党': 'right',
    '日本維新の党': 'right',
    '参政党': 'right',
    '日本保守党': 'right',
    '中道': 'center',
}

ideology_colors = {'left': '#E74C3C', 'center': '#9B59B6', 'right': '#3498DB'}

print("✓ イデオロギー分類を設定")

In [ ]:
# Plot A: 政党政策マップ（Plotly バブルチャート）
print("\nCreating Plot A (Party Policy Map)...")

party_info = []
for party, umap_coords in zip(party_list, umap_parties):
    chamber_list = df_paragraphs[df_paragraphs['party'] == party]['chamber'].unique().tolist()
    total_seats = party_seats.get(party, 0)
    ideology = ideology_map.get(party, 'center')
    color = ideology_colors[ideology]

    party_info.append({
        'party': party,
        'umap_x': umap_coords[0],
        'umap_y': umap_coords[1],
        'seats': max(total_seats, 1),  # 0席の場合は1にして表示
        'ideology': ideology,
        'chambers': ', '.join(chamber_list),
        'color': color,
    })

df_parties = pd.DataFrame(party_info)

fig_a = px.scatter(
    df_parties,
    x='umap_x',
    y='umap_y',
    size='seats',
    color='ideology',
    hover_name='party',
    hover_data={'umap_x': False, 'umap_y': False, 'seats': True, 'chambers': True, 'ideology': False},
    color_discrete_map={'left': '#E74C3C', 'center': '#9B59B6', 'right': '#3498DB'},
    size_max=60,
    title='政党政策マップ（議席数=バブルサイズ）',
    labels={'umap_x': '政策軸1', 'umap_y': '政策軸2', 'ideology': 'イデオロギー'},
    height=700,
)

# テキストラベルを追加
fig_a.add_scatter(
    x=df_parties['umap_x'],
    y=df_parties['umap_y'],
    text=df_parties['party'],
    mode='text',
    textposition='top center',
    textfont=dict(size=9),
    showlegend=False,
    hoverinfo='skip',
)

fig_a.update_layout(
    font=dict(family='Arial', size=12),
    template='plotly_white',
)

fig_a.write_html(os.path.join(results_path, 'plot_A_party_map.html'))
fig_a.show()

print(f"✓ Plot A を {results_path}/plot_A_party_map.html に保存")

In [ ]:
# Plot B: トピック地形図（パラグラフ散布図）
print("Creating Plot B (Topic Landscape)...")

df_plot_b = df_paragraphs.copy()
df_plot_b['cluster_label'] = df_plot_b['cluster'].map(
    {i: topic_labels[i] for i in range(n_topics)}
)

fig_b = px.scatter(
    df_plot_b,
    x='umap_x',
    y='umap_y',
    color='cluster_label',
    hover_data={'umap_x': False, 'umap_y': False, 'party': True, 'chamber': True, 'text': True},
    opacity=0.5,
    title='政策トピック地形図（パラグラフレベル）',
    labels={'umap_x': '政策軸1', 'umap_y': '政策軸2', 'cluster_label': 'トピック'},
    height=700,
)

fig_b.update_layout(
    font=dict(family='Arial', size=10),
    template='plotly_white',
    hovermode='closest',
)

fig_b.write_html(os.path.join(results_path, 'plot_B_topic_landscape.html'))
fig_b.show()

print(f"✓ Plot B を {results_path}/plot_B_topic_landscape.html に保存")

In [ ]:
# Plot C: トピック×政党ヒートマップ
print("Creating Plot C (Topic-Party Heatmap)...")

# 政党を左右順に並び替え
party_order = [
    # 左
    '共産党', '立憲民主党', '社民党', '令和新選組',
    # 中道
    '公明党', '国民民主党', 'チームみらい',
    # 右
    '自民党', '中道', '日本維新の党', '参政党', '日本保守党',
]

# データフレームを政党順に並び替え
party_topic_ordered = party_topic_matrix.loc[[p for p in party_order if p in party_topic_matrix.index]]

# ヒートマップ
plt.figure(figsize=(14, 8))
sns.heatmap(
    party_topic_ordered,
    cmap='RdYlBu_r',
    annot=True,
    fmt='.2f',
    cbar_kws={'label': 'NMF Weight'},
    linewidths=0.5,
)

plt.title('トピック-政党行列（NMF重み）', fontsize=14, fontweight='bold')
plt.xlabel('トピック', fontsize=12)
plt.ylabel('政党', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(results_path, 'plot_C_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Plot C を {results_path}/plot_C_heatmap.png に保存")

In [ ]:
# Plot D: 超党派重要度ランキング
print("Creating Plot D (Cross-Party Importance Ranking)...")

plot_d_data = topic_scores.sort_values('salience_weighted', ascending=True)

plt.figure(figsize=(12, 8))
bars = plt.barh(
    plot_d_data['topic_label'],
    plot_d_data['salience_weighted'],
    color='#3498DB'
)

# 出現政党数を右側にアノテーション
for idx, (topic, score, party_count) in enumerate(zip(
    plot_d_data['topic_label'],
    plot_d_data['salience_weighted'],
    plot_d_data['party_count']
)):
    plt.text(score + 0.005, idx, f'{party_count}/19政党', va='center', fontsize=9)

plt.xlabel('議席数加重スコア', fontsize=12)
plt.title('超党派重要度ランキング（議席数加重）', fontsize=14, fontweight='bold')
plt.xlim(0, plot_d_data['salience_weighted'].max() * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(results_path, 'plot_D_ranking.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Plot D を {results_path}/plot_D_ranking.png に保存")

## Section 6: 解析結果の解釈

In [ ]:
print("\n" + "="*80)
print("超党派政策解析 - 主要な知見")
print("="*80)

print("\n【1】最も超党派的に重視されているテーマ (top 5)")
top_5 = topic_scores.head(5)
for idx, row in top_5.iterrows():
    print(f"  {row['topic_label']:15} (スコア: {row['salience_weighted']:.3f}, 出現政党: {int(row['party_count'])}/19)")

print("\n【2】政党別の政策特性")
print("\n左派:")
for party in ['共産党', '立憲民主党', '社民党', '令和新選組']:
    if party in party_topic_matrix.index:
        top_topics = party_topic_matrix.loc[party].nlargest(2)
        print(f"  {party}: {', '.join(top_topics.index.tolist())}")

print("\n中道:")
for party in ['公明党', '国民民主党', 'チームみらい']:
    if party in party_topic_matrix.index:
        top_topics = party_topic_matrix.loc[party].nlargest(2)
        print(f"  {party}: {', '.join(top_topics.index.tolist())}")

print("\n右派:")
for party in ['自民党', '日本維新の党', '参政党']:
    if party in party_topic_matrix.index:
        top_topics = party_topic_matrix.loc[party].nlargest(2)
        print(f"  {party}: {', '.join(top_topics.index.tolist())}")

print("\n【3】統計サマリー")
print(f"  分析対象政党: {df_paragraphs['party'].nunique()}")
print(f"  総パラグラフ数: {len(df_paragraphs)}")
print(f"  抽出トピック数: {n_topics}")
print(f"  K-Means シルエットスコア: {silhouette:.3f}")
print(f"\n✓ 解析完了 - 結果は {results_path} に保存されました")

In [ ]:
print("\n【出力ファイル一覧】")
output_files = [
    'ranked_topics.csv',
    'party_topic_matrix.csv',
    'paragraph_clusters.csv',
    'paragraph_embeddings.npy',
    'plot_A_party_map.html',
    'plot_B_topic_landscape.html',
    'plot_C_heatmap.png',
    'plot_D_ranking.png',
]

for fname in output_files:
    fpath = os.path.join(results_path, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024  # KB
        print(f"  ✓ {fname:40} ({size:.1f} KB)")
    else:
        print(f"  ✗ {fname:40} (未生成)")